## ToolStrategy使用详解
ToolStrategy通过`工具调用`（Tool Calling/Function Calling）实现结构化输出，所以LangChain会在消息列表末尾`追加一条ToolMessage`，让整个链路完整。但实际上没有实际的工具执行，这是一条伪消息。

ToolStrategy适用于任何支持工具调用的现代模型。

ToolStrategy的配置包含三个主要参数：
- schema（必需参数）：与提供商策略的schema参数功能一致，支持`Pydantic模型`、`TypedDict`、`JSON Schema`、`数据类(@dataclass)`，同时还支持联合类型`Union[类型1, 类型2]`（允许模型根据输入内容选择最匹配的数据结构）。
- tool_message_content（可选参数）：用于自定义生成结构化输出时，会话历史中记录的提示信息。默认使用展示输出数据的标准响应语句。
- handle_errors（可选参数）：用于指定数据校验失败时的重试策略，`默认值为True`。

### 1.结构化输出：schema参数
输出模式1：Pydantic类型(最推荐的类型)

In [2]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

agent = create_agent(
    model=model_openai,
    response_format=ToolStrategy(ContactInfo),
)

response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：boy12@hallo.com，手机号：13989892121")
    ]
})

rprint(response)

# for msg in response["messages"]:
#     msg.pretty_print()


{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：boy12@hallo.com，手机号：13989892121',
            additional_kwargs={},
            response_metadata={},
            id='938a53e2-05b0-400a-9875-96fda9919ca2'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 383,
                    'prompt_tokens': 344,
                    'total_tokens': 727,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 316,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 383
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'text_tokens': 344
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-0153f68c-169d-9112-8157-579b901f3b15',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0bf23-1962-7813-9149-fb9adf1eec77-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'boy12@hallo.com', 'phone': '13989892121'},
                    'id': 'call_288a6daac6c04a72bce0bbe5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 344,
                'output_tokens': 383,
                'total_tokens': 727,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 316}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='boy12@hallo.com' phone='13989892121'",
            name='ContactInfo',
            id='5149c89b-6e58-4b64-826b-1e3b15e1bc6d',
            tool_call_id='call_288a6daac6c04a72bce0bbe5'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='boy12@hallo.com', phone='13989892121')
}

举例2：

In [8]:
from langchain.messages import SystemMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import Literal
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query: 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer: 客户名称，例如 "张三" 或 "李四"

    Returns:
        确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

#定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户", description="客户等级，只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低","中","高"] = Field(None, description="消费水平")
    sent_email: bool = Field(False, description="是否已经发送感谢邮件")

#创建智能体
agent = create_agent(
    model=model_openai,
    system_prompt=SystemMessage(content=""
                               "请分析指定客户的情况："
                                "1. 先搜索客户数据库了解最新情况 "
                                "2. 如果是VIP客户，则发送感谢邮件 "
                                "3. 基于搜索结果生成结构化分析报告 "
                                "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis),
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "请分析客户张三"}
    ]
})

if "structured_response" in result:
    analysis = result["structured_response"]
    print(analysis)


customer_name='张三' customer_tier='VIP客户' recent_activity='2026-01-15' spending_level='高' sent_email=True


输出模式2：TypedDict类型

In [9]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from typing import TypedDict, Annotated
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

class ContactInfo(TypedDict):
    """用户的联系方式"""
    name: Annotated[str, ..., "用户姓名"]
    email: Annotated[str, ..., "用户邮箱地址"]
    phone: Annotated[str, ..., "用户的手机号"]

agent = create_agent(
    model=model_openai,
    response_format=ToolStrategy(ContactInfo),
)

response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：hellox@qq.com，手机号：14731231031")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：hellox@qq.com，手机号：14731231031
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_c704a2aa79dd46689e93f155)
 Call ID: call_c704a2aa79dd46689e93f155
  Args:
    name: 小明
    email: hellox@qq.com
    phone: 14731231031
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: {'name': '小明', 'email': 'hellox@qq.com', 'phone': '14731231031'}


In [13]:
from langchain.messages import SystemMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal, Optional
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query: 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer: 客户名称，例如 "张三" 或 "李四"

    Returns:
        确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

#定义TypedDict Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], None, "客户姓名"]
    customer_tier: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"], None, "潜在客户"]
    recent_activity: Annotated[Optional[str], None, "最近活动"]
    spending_level: Annotated[Literal["低","中","高"], None, "消费水平"]
    sent_email: Annotated[bool, False, "是否已经发送感谢邮件"]

#创建智能体
agent = create_agent(
    model=model_openai,
    system_prompt=SystemMessage(content=""
                               "请分析指定客户的情况："
                                "1. 先搜索客户数据库了解最新情况 "
                                "2. 如果是VIP客户，则发送感谢邮件 "
                                "3. 基于搜索结果生成结构化分析报告 "
                                "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis),
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "请分析客户张三"}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

请分析客户张三
================================== Ai Message ==================================
Tool Calls:
  search_customer_database (call_6f5a700b7a1d463f97fe6da1)
 Call ID: call_6f5a700b7a1d463f97fe6da1
  Args:
    query: 张三
================================= Tool Message =================================
Name: search_customer_database

客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000
================================== Ai Message ==================================
Tool Calls:
  send_email (call_910774bb844d4301b3c6f368)
 Call ID: call_910774bb844d4301b3c6f368
  Args:
    customer: 张三
  CustomerAnalysis (call_987abc844a414fafb329ef48)
 Call ID: call_987abc844a414fafb329ef48
  Args:
    customer_name: 张三
    customer_tier: VIP客户
    recent_activity: 2026-01-15
    spending_level: 高
    sent_email: True
================================= Tool Message =================================
Name: CustomerAnalysis

Returning

输出模式3：JsonSchema类型

In [14]:
from langchain.messages import SystemMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query: 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer: 客户名称，例如 "张三" 或 "李四"

    Returns:
        确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

# 定义 JSON Schema 替代 Pydantic 模型
customer_analysis_schema = {
    "title": "CustomerAnalysis",
    "type": "object",
    "description": "客户分析报告",
    "properties": {
        "customer_name": {
            "type": "string",
            "default": "",
            "description": "客户姓名"
        },
        "customer_tier": {
            "type": "string",
            "enum": ["潜在客户", "普通客户", "VIP客户", "流失风险"],
            "default": "潜在客户",
            "description": "客户等级"
        },
        "recent_activity": {
            "type": "string",
            "default": "",
            "description": "最近活动"
        },
        "spending_level": {
            "type": "string",
            "enum": ["低", "中", "高"],
            "default": "低",
            "description": "消费水平"
        },
        "send_email": {
            "type": "boolean",
            "default": False,
            "description": "是否已发送感谢邮件"
        }
    },
    # 所有字段都是必须输出的
    "required": ["customer_name", "customer_tier", "recent_activity","spending_level"]
}
#创建智能体
agent = create_agent(
    model=model_openai,
    system_prompt=SystemMessage(content=""
                               "请分析指定客户的情况："
                                "1. 先搜索客户数据库了解最新情况 "
                                "2. 如果是VIP客户，则发送感谢邮件 "
                                "3. 基于搜索结果生成结构化分析报告 "
                                "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(customer_analysis_schema),
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "请分析客户张三"}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

请分析客户张三
================================== Ai Message ==================================
Tool Calls:
  search_customer_database (call_99a63acea4704533a683abcc)
 Call ID: call_99a63acea4704533a683abcc
  Args:
    query: 张三
================================= Tool Message =================================
Name: search_customer_database

客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000
================================== Ai Message ==================================
Tool Calls:
  send_email (call_7f9a058c85974b5e86dd10f4)
 Call ID: call_7f9a058c85974b5e86dd10f4
  Args:
    customer: 张三
  CustomerAnalysis (call_0a65dfeb1fe142efaddaeda2)
 Call ID: call_0a65dfeb1fe142efaddaeda2
  Args:
    customer_name: 张三
    customer_tier: VIP客户
    recent_activity: 最近购买日期：2026-01-15
    spending_level: 高
    send_email: True
================================= Tool Message =================================
Name: CustomerAnalysis

Re

输出模式4：@dataclass类型

In [15]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dataclasses import dataclass
from rich import print as rprint

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@dataclass
class ContactInfo:
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

agent = create_agent(
    model=model_openai,
    response_format=ToolStrategy(ContactInfo),
)

response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：boy12@hallo.com，手机号：13989892121")
    ]
})

rprint(response)

# for msg in response["messages"]:
#     msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：boy12@hallo.com，手机号：13989892121',
            additional_kwargs={},
            response_metadata={},
            id='f22c4d01-1d79-4cc5-9c7f-631325ff322c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 288,
                    'prompt_tokens': 344,
                    'total_tokens': 632,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 221,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 288
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'text_tokens': 344
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-756302a2-a1a8-9ced-bb12-b0c51e53a4c8',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0bf87-e0a1-72e2-81af-cd02064476a2-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'boy12@hallo.com', 'phone': '13989892121'},
                    'id': 'call_08ce6c34bdfb48dab1702423',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 344,
                'output_tokens': 288,
                'total_tokens': 632,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 221}
            }
        ),
        ToolMessage(
            content="Returning structured response: ContactInfo(name='小明', email='boy12@hallo.com', 
phone='13989892121')",
            name='ContactInfo',
            id='738e6407-e34b-4961-8c32-509ad7086de8',
            tool_call_id='call_08ce6c34bdfb48dab1702423'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='boy12@hallo.com', phone='13989892121')
}